In [ ]:
#Scrapping of speeches before 2011
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re 

all_years_data = []

for year in range(2005, 2010):
    # Fetching the webpage for the current year
    page = requests.get(f'https://www2.boj.or.jp/archive/en/announcements/press/koen_{year}/index.htm')
    soup = BeautifulSoup(page.text, 'html.parser')

    for i in range(1, 19):  # Change the range according to the number of rows you want to iterate over
        selector = f'#contents-skip > div > table.js-tbl > tbody > tr:nth-child({i}) > td:nth-child(2) > a'

        links = soup.select(selector)

        if links:
            link = 'https://www2.boj.or.jp' + links[0]['href']
            all_years_data.append({'link': link, 'year': year})
        else:
            print(f'No link found for row {i}')

all_years = pd.DataFrame(all_years_data)
print(all_years)

In [5]:
for i in range(len(emails)):
    print(f"Processing row {i}")
    try:
        page = requests.get(all_years.loc[i, 'Link'])
        page.raise_for_status()  # Checking if the request was successful

        soup = BeautifulSoup(page.text, 'html.parser')

        # Find all 'p' tags
        p_tags = soup.find_all('p')

        # Extract text from each 'p' tag
        text_list = []
        for p_tag in p_tags:
            text_list.append(p_tag.get_text())

        # Combine the text from all 'p' tags
        combined_text = ' '.join(text_list)

        # Remove unwanted characters
        combined_text = combined_text.replace('--', ' ')
        combined_text = combined_text.replace('\r', '')
        combined_text = combined_text.replace('\t', '')

        emails.loc[i, 'text'] = combined_text

    except requests.exceptions.RequestException as e:
        # Handle request exceptions
        print(f"Error processing row {i}: {e}")
        emails.loc[i, 'text'] = "" 

    # After processing, checking the 'text' column for the specified row
    if not emails.loc[i, 'text']:
        print(f"The 'text' column for row {i} is empty.")
    else:
        print(f"The 'text' column for row {i} is not empty.")


Processing row 0
The 'text' column for row 0 is not empty.
Processing row 1
The 'text' column for row 1 is not empty.
Processing row 2
The 'text' column for row 2 is not empty.
Processing row 3
The 'text' column for row 3 is not empty.
Processing row 4
The 'text' column for row 4 is not empty.
Processing row 5
The 'text' column for row 5 is not empty.
Processing row 6
The 'text' column for row 6 is not empty.
Processing row 7
The 'text' column for row 7 is not empty.
Processing row 8
The 'text' column for row 8 is not empty.
Processing row 9
The 'text' column for row 9 is not empty.
Processing row 10
The 'text' column for row 10 is not empty.
Processing row 11
The 'text' column for row 11 is not empty.
Processing row 12
The 'text' column for row 12 is not empty.
Processing row 13
The 'text' column for row 13 is not empty.
Processing row 14
The 'text' column for row 14 is not empty.
Processing row 15
The 'text' column for row 15 is not empty.
Processing row 16
The 'text' column for row 

In [9]:
row_to_check = 7

# Printing the text content for the specified row
print(emails.loc[row_to_check, 'text'])


Archives 
Home > 
        About the Bank > 
        Speeches and Statements > 
        Speeches 1996–2010 > Speeches 2005 > Recent Economic and Financial Developments in Japan
       Summary of a speech given by Toshikatsu Fukuma, Member of the Policy Board, at a meeting with business leaders in Fukui on September 14, 2005 December 8, 2005Bank of Japan I would like to begin first with the recent economic and financial picture, starting with developments overseas.  There is economic expansion worldwide.  Particularly noticeable is the rapid economic growth of China, a cornerstone of the so-called BRICs.  International trade is growing every year supported by this worldwide economic expansion.  Strong increases in energy sources and raw materials, such as crude oil, iron ore, and steel, and in machinery, are the main contributing factors.  This indicates that demand for goods essential for the development of social infrastructure and natural resources, and for industrialization such as s

In [11]:
emails.to_excel('boj_speeches_2005_2010.xlsx',index=False)

In [4]:
import gensim
from gensim import corpora
from gensim.models import LdaModel
from gensim.models.ldamulticore import LdaMulticore
from gensim.utils import simple_preprocess
import pandas as pd
from gensim.utils import simple_preprocess
from nltk.corpus import stopwords

def preprocess_text(text):
    stop_words = set(stopwords.words('english'))
    
    # Convert non-string values to string
    if not isinstance(text, str):
        text = str(text)
        
    tokens = simple_preprocess(text)
    tokens = [token for token in tokens if token not in stop_words]
    return tokens

speeches_df = pd.read_excel(r'C:\Users\user\Desktop\BOJdata\BOJdatasentimentfinalmv.xlsx')

# Preprocessing speeches
speeches_df['processed_text'] = speeches_df['text'].apply(preprocess_text)

# Creating dictionary and corpus
dictionary = corpora.Dictionary(speeches_df['processed_text'])
corpus = [dictionary.doc2bow(text) for text in speeches_df['processed_text']]

# Train LDA model
lda_model = LdaMulticore(corpus=corpus,
                          id2word=dictionary,
                          num_topics=10, 
                          workers=4,
                          passes=10)

# Function to get topic distribution for a speech
def get_topic_distribution(text):
    bow = dictionary.doc2bow(preprocess_text(text))
    topic_distribution = lda_model.get_document_topics(bow)
    return topic_distribution


In [5]:
for topic_idx, topic in lda_model.print_topics():
    print(topic)
    print()

0.025*"financial" + 0.021*"bank" + 0.015*"payment" + 0.012*"service" + 0.012*"system" + 0.011*"settlement" + 0.011*"market" + 0.009*"central" + 0.007*"new" + 0.007*"japan"

0.018*"financial" + 0.016*"bank" + 0.015*"economy" + 0.011*"economic" + 0.011*"growth" + 0.010*"japan" + 0.009*"rate" + 0.008*"firm" + 0.008*"price" + 0.007*"institution"

0.019*"price" + 0.018*"economy" + 0.012*"economic" + 0.012*"rate" + 0.010*"bank" + 0.010*"firm" + 0.010*"japan" + 0.009*"policy" + 0.008*"monetary" + 0.008*"financial"

0.021*"price" + 0.015*"rate" + 0.014*"percent" + 0.014*"bank" + 0.013*"economy" + 0.011*"inflation" + 0.010*"increase" + 0.009*"japan" + 0.008*"policy" + 0.008*"economic"

0.012*"policy" + 0.008*"growth" + 0.006*"bank" + 0.006*"public" + 0.006*"good" + 0.006*"would" + 0.005*"view" + 0.005*"financial" + 0.005*"aging" + 0.005*"social"

0.018*"economy" + 0.017*"market" + 0.016*"financial" + 0.011*"ha" + 0.011*"economic" + 0.010*"growth" + 0.010*"japan" + 0.010*"global" + 0.009*"asian"

In [13]:
import pandas as pd

topics_of_interest = [2, 8] 

# Creating an empty list to store relevant speeches
relevant_speeches = []

# Iterating through each speech and filter based on topics of interest
for idx, speech in speeches_df.iterrows():
    topic_distribution = get_topic_distribution(speech['text'])
    relevant_topics = [topic for topic, prob in topic_distribution if topic in topics_of_interest and prob > 0.2]
    if relevant_topics:
        relevant_speeches.append(speech)

relevant_speeches_df = pd.DataFrame(relevant_speeches)

print(relevant_speeches_df)

     Unnamed: 0.1  Unnamed: 0           Date  \
41           41.0        42.0  Oct. 10, 2012   
61           61.0        64.0  Feb. 17, 2012   
79           79.0        82.0  Oct. 10, 2013   
80           80.0        83.0  Oct. 10, 2013   
81           81.0        84.0  Oct.  9, 2013   
..            ...         ...            ...   
410           NaN         NaN  Jan. 10, 2008   
416           NaN         NaN  Dec.  2, 2009   
426           NaN         NaN  Sep.  9, 2009   
450           NaN         NaN  Dec.  1, 2010   
473           NaN         NaN  Apr. 21, 2010   

                                Speaker  \
41   WAKATABE Masazumi, Deputy Governor   
61   WAKATABE Masazumi, Deputy Governor   
79   WAKATABE Masazumi, Deputy Governor   
80   WAKATABE Masazumi, Deputy Governor   
81   WAKATABE Masazumi, Deputy Governor   
..                                  ...   
410                                 NaN   
416                                 NaN   
426                                 

In [14]:
relevant_speeches_df.to_excel('BoJdatasentimentfutureintpart3.xlsx', index=False)